<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/05_Analyse_Radiographies_Thoraciques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Analyse de Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

---

## Objectifs

- Utiliser TorchXRayVision pour détecter des pathologies
- Charger des images depuis datasets ou upload
- Interpréter les résultats

## Pathologies Détectables (18)

Atélectasie, Cardiomégalie, Épanchement, Infiltration, Masse, Nodule, Pneumonie, Pneumothorax, Consolidation, Œdème, Emphysème, Fibrose, Épaississement pleural, Hernie, Opacité pulmonaire, Lésion pleurale, Fracture, Dispositif médical

## 1. Installation

In [ ]:
!pip install -q torchxrayvision torch torchvision scikit-image pillow matplotlib

In [ ]:
import torchxrayvision as xrv
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import skimage.io
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Chargement du Modèle

In [ ]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model = model.to(device)
model.eval()

print(f"\nPathologies ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies, 1):
    print(f"{i:2d}. {p}")

## 3. Datasets Disponibles

TorchXRayVision inclut plusieurs datasets:

In [ ]:
print("Datasets disponibles:")
print("  - NIH_Google_Dataset (petit, idéal pour tests)")
print("  - NIH_Dataset (112k images)")
print("  - CheX_Dataset (224k images)")
print("  - MIMIC_Dataset (377k images)")
print("  - PC_Dataset (PadChest, 160k images)")
print("\nNous utiliserons un échantillon ou upload.")

## 4. Chargement d'une Image

### Option A: Upload

In [ ]:
from google.colab import files

uploaded = files.upload()
if uploaded:
    img_path = list(uploaded.keys())[0]
    print(f"Image: {img_path}")

### Option B: Depuis Dataset

In [ ]:
# Exemple avec NIH_Google (décommentez si dataset disponible)
# dataset = xrv.datasets.NIH_Google_Dataset(
#     imgpath="./images",
#     csvpath="./nih_google.csv",
#     transform=torchvision.transforms.Compose([
#         xrv.datasets.XRayCenterCrop(),
#         xrv.datasets.XRayResizer(224)
#     ])
# )
# sample = dataset[0]
# img_tensor = sample['img'].unsqueeze(0).to(device)

## 5. Préparation de l'Image

In [ ]:
def prepare_image(img_path):
    # Charger
    img = skimage.io.imread(img_path)
    
    # Normaliser ([-1024, 1024])
    img = xrv.datasets.normalize(img, 255)
    
    # Convertir en grayscale si nécessaire
    if len(img.shape) == 3:
        img = img.mean(2)
    img = img[None, ...]
    
    # Transformer
    transform = torchvision.transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224)
    ])
    img = transform(img)
    
    # Tensor
    img_tensor = torch.from_numpy(img).to(device)
    return img_tensor, img[0]

img_tensor, img_display = prepare_image(img_path)

plt.figure(figsize=(6, 6))
plt.imshow(img_display, cmap='gray')
plt.title("Image Préparée")
plt.axis('off')
plt.show()

print(f"Shape: {img_tensor.shape}")

## 6. Analyse

In [ ]:
with torch.no_grad():
    pred = model(img_tensor[None,...])
    probs = torch.sigmoid(pred).cpu().numpy()[0]

# Trier
indices = np.argsort(probs)[::-1]

print("RÉSULTATS")
print("="*50)
for idx in indices[:10]:
    pathology = model.pathologies[idx]
    prob = probs[idx]
    bar = '█' * int(prob * 20)
    print(f"{pathology:<25} {prob:>6.1%} {bar}")
print("="*50)

## 7. Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Image
ax1.imshow(img_display, cmap='gray')
ax1.set_title("Radiographie")
ax1.axis('off')

# Barres
top_n = 8
top_indices = indices[:top_n]
top_pathologies = [model.pathologies[i] for i in top_indices]
top_probs = [probs[i] for i in top_indices]

colors = ['red' if p > 0.5 else 'orange' if p > 0.3 else 'steelblue' for p in top_probs]
ax2.barh(range(top_n), top_probs, color=colors)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(top_pathologies)
ax2.set_xlabel('Probabilité')
ax2.set_title('Top 8 Pathologies')
ax2.set_xlim(0, 1)
ax2.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
ax2.axvline(x=0.3, color='orange', linestyle='--', alpha=0.5)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Interprétation

In [ ]:
anomalies_probables = [(model.pathologies[i], probs[i]) for i in range(len(probs)) if probs[i] > 0.5]
anomalies_possibles = [(model.pathologies[i], probs[i]) for i in range(len(probs)) if 0.3 < probs[i] <= 0.5]

print("\nINTERPRÉTATION")
print("="*50)

if anomalies_probables:
    print("\nAnomalies PROBABLES (>50%):")
    for patho, prob in anomalies_probables:
        print(f"  - {patho}: {prob:.1%}")

if anomalies_possibles:
    print("\nAnomalies POSSIBLES (30-50%):")
    for patho, prob in anomalies_possibles:
        print(f"  - {patho}: {prob:.1%}")

if not anomalies_probables and not anomalies_possibles:
    print("\nAucune anomalie majeure détectée.")

print("\n" + "="*50)
print("ATTENTION: Validation par radiologue obligatoire.")
print("="*50)

## 9. Points Clés

### Ce que vous avez appris:
- Charger un modèle TorchXRayVision
- Analyser des radiographies (18 pathologies)
- Interpréter les probabilités

### Applications:
- **Aide au diagnostic**: Détection d'anomalies
- **Triage**: Prioriser cas urgents
- **Recherche**: Analyser grandes cohortes

### Limites:
- Ne remplace PAS le radiologue
- Sensible à qualité d'image
- Peut manquer pathologies rares
- Validation clinique requise